In [ ]:
%load_ext autoreload
%autoreload 2

import mofaflex as mfl
import matplotlib.pyplot as plt
from plotnine import *
import pandas as pd
import numpy as np
import anndata as ad
import scanpy as sc
import pickle as pkl
import h5py
import utils
from io import StringIO
import requests
import networkx as nx

## Load Data

In [ ]:
adata = ad.read_h5ad(f"data/anndata.h5ad", backed="r")
adata = ad.AnnData(obs=adata.obs, var=adata.var, varm=adata.varm, obsm=adata.obsm)
adata = adata[adata.obs.pathway != "INS"]
adata.obs_names_make_unique()

## Get Factors and Loadings

### MOFAFLEX

In [ ]:
mfl_model = mfl.MOFAFLEX.load(f"models/mfl_complete.h5")

mfl_factors = pd.concat(mfl_model.get_factors("pandas"))
mfl_factors.index = mfl_factors.index.droplevel(0)
mfl_factors = mfl_factors.loc[adata.obs_names]
mfl_loadings = mfl_model.get_weights("pandas")["RNA"]

In [ ]:
# subset anndata to used genes
adata = adata[:, mfl_model.feature_names["RNA"]]

mask = pd.read_csv("hallmark_mask.csv", index_col=0)
mask = mask.loc[adata.var_names]
mask = mask.loc[:, mask.sum(axis=0) >= 5]

adata.varm["annotations"] = mask

### Spectra

In [ ]:
spectra_dict = pkl.load(open(f"models/spectra_complete.pkl", "rb"))

spectra_factors = pd.DataFrame(spectra_dict["cell_scores"])
spectra_loadings = pd.DataFrame(spectra_dict["factors"])

spectra_factors.columns = [f"Factor {i}" for i in range(1, spectra_factors.shape[1]+1)]
spectra_factors.index = adata.obs_names

spectra_loadings.index = spectra_factors.columns
spectra_loadings.columns = adata.var_names

# assign gene sets to factors based on overlap of Spectra factor markers
spectra_gs_assignment = utils.assign_markers_to_gene_sets(
    spectra_dict["SPECTRA_markers"], mask
)
# if duplicates are present, keep assignment with highest overlap
spectra_gs_assignment = spectra_gs_assignment.sort_values('overlap_coefficient', ascending=False).drop_duplicates('gene_set').sort_index()

# remove unassigned factors
spectra_gs_assignment = spectra_gs_assignment[spectra_gs_assignment.overlap_coefficient > 0.2]

spectra_factors = spectra_factors[spectra_gs_assignment.factor]
spectra_factors.columns = spectra_gs_assignment.gene_set

spectra_loadings = spectra_loadings.loc[spectra_gs_assignment.factor]
spectra_loadings.index = spectra_gs_assignment.gene_set


### ExpiMap

In [ ]:
expimap_dict = pkl.load(open(f"models/expimap_complete.pkl", "rb"))

expimap_factors = pd.DataFrame(expimap_dict["latents"])
expimap_factors.columns = mask.columns
expimap_factors.index = expimap_dict["obs_names"]

expimap_loadings = pd.DataFrame(expimap_dict["weights"], index=mask.columns, columns=expimap_dict["var_names"])

## Prettify Factor Names

In [ ]:
hallmark_map = {
    'HALLMARK_ADIPOGENESIS': 'Adipogenesis',
    'HALLMARK_ALLOGRAFT_REJECTION': 'Allograft Rejection',
    'HALLMARK_ANDROGEN_RESPONSE': 'Androgen Response',
    'HALLMARK_ANGIOGENESIS': 'Angiogenesis',
    'HALLMARK_APICAL_JUNCTION': 'Apical Junction',
    'HALLMARK_APICAL_SURFACE': 'Apical Surface',
    'HALLMARK_APOPTOSIS': 'Apoptosis',
    'HALLMARK_BILE_ACID_METABOLISM': 'Bile Acid Metabolism',
    'HALLMARK_CHOLESTEROL_HOMEOSTASIS': 'Cholesterol Homeostasis',
    'HALLMARK_COAGULATION': 'Coagulation',
    'HALLMARK_COMPLEMENT': 'Complement',
    'HALLMARK_DNA_REPAIR': 'DNA Repair',
    'HALLMARK_E2F_TARGETS': 'E2F Targets',
    'HALLMARK_EPITHELIAL_MESENCHYMAL_TRANSITION': 'Epithelial-Mesenchymal Transition',
    'HALLMARK_ESTROGEN_RESPONSE_EARLY': 'Estrogen Response (Early)',
    'HALLMARK_ESTROGEN_RESPONSE_LATE': 'Estrogen Response (Late)',
    'HALLMARK_FATTY_ACID_METABOLISM': 'Fatty Acid Metabolism',
    'HALLMARK_G2M_CHECKPOINT': 'G2M Checkpoint',
    'HALLMARK_GLYCOLYSIS': 'Glycolysis',
    'HALLMARK_HEDGEHOG_SIGNALING': 'Hedgehog Signaling',
    'HALLMARK_HEME_METABOLISM': 'Heme Metabolism',
    'HALLMARK_HYPOXIA': 'Hypoxia',
    'HALLMARK_IL2_STAT5_SIGNALING': 'IL2/STAT5 Signaling',
    'HALLMARK_IL6_JAK_STAT3_SIGNALING': 'IL6/JAK/STAT3 Signaling',
    'HALLMARK_INFLAMMATORY_RESPONSE': 'Inflammatory Response',
    'HALLMARK_INTERFERON_ALPHA_RESPONSE': 'Interferon Alpha Response',
    'HALLMARK_INTERFERON_GAMMA_RESPONSE': 'Interferon Gamma Response',
    'HALLMARK_KRAS_SIGNALING_DN': 'KRAS Signaling (Down)',
    'HALLMARK_KRAS_SIGNALING_UP': 'KRAS Signaling (Up)',
    'HALLMARK_MITOTIC_SPINDLE': 'Mitotic Spindle',
    'HALLMARK_MTORC1_SIGNALING': 'mTORC1 Signaling',
    'HALLMARK_MYOGENESIS': 'Myogenesis',
    'HALLMARK_P53_PATHWAY': 'p53 Pathway',
    'HALLMARK_PANCREAS_BETA_CELLS': 'Pancreas Beta Cells',
    'HALLMARK_PEROXISOME': 'Peroxisome',
    'HALLMARK_PI3K_AKT_MTOR_SIGNALING': 'PI3K/AKT/mTOR Signaling',
    'HALLMARK_REACTIVE_OXYGEN_SPECIES_PATHWAY': 'Reactive Oxygen Species',
    'HALLMARK_SPERMATOGENESIS': 'Spermatogenesis',
    'HALLMARK_TGF_BETA_SIGNALING': 'TGF-Beta Signaling',
    'HALLMARK_TNFA_SIGNALING_VIA_NFKB': 'TNF Alpha Signaling via NF-kB',
    'HALLMARK_UNFOLDED_PROTEIN_RESPONSE': 'Unfolded Protein Response',
    'HALLMARK_UV_RESPONSE_DN': 'UV Response (Down)',
    'HALLMARK_UV_RESPONSE_UP': 'UV Response (Up)',
    'HALLMARK_WNT_BETA_CATENIN_SIGNALING': 'WNT/Beta-Catenin Signaling',
    'HALLMARK_XENOBIOTIC_METABOLISM': 'Xenobiotic Metabolism',
    'HALLMARK_NOTCH_SIGNALING': 'Notch Signaling',
    'HALLMARK_PROTEIN_SECRETION': 'Protein Secretion'
}

## Explained Variance

In [ ]:
df_r2 = mfl_model.get_r2()
consensus_df = utils.consensus_ranking(df_r2)

top_n = 20

# Build rank matrix: rows = top factors (by consensus), cols = cell types + Mean Rank
rank_cols = [c for c in consensus_df.columns if c not in ("mean_rank", "consensus_rank")]
filtered_df = consensus_df[~consensus_df.index.str.startswith("Factor")]
heat_df = filtered_df.head(top_n)[rank_cols].copy()
heat_df["Mean Rank"] = filtered_df.head(top_n)["mean_rank"]

# Apply hallmark_map to factor index
heat_df.index = heat_df.index.map(lambda x: hallmark_map.get(x, x))

# Melt to long form
heat_long = (
    heat_df
    .reset_index(names="Factor")
    .melt(id_vars="Factor", var_name="Group", value_name="Rank")
)

# Factor order: top consensus factor at left (landscape)
factor_order = heat_df.index.tolist()
heat_long["Factor"] = pd.Categorical(heat_long["Factor"], categories=factor_order, ordered=True)

# Group order: cell types alphabetically, Mean Rank last
group_order = sorted(rank_cols) + ["Mean Rank"]
heat_long["Group"] = pd.Categorical(heat_long["Group"], categories=group_order, ordered=True)
heat_long["is_consensus"] = heat_long["Group"] == "Mean Rank"

plot_ranks = (
    ggplot(heat_long, aes(x="Factor", y="Group", fill="Rank"))
    + geom_tile(aes(color="is_consensus"), size=0.3)
    + geom_text(aes(label="Rank.round().astype(int).astype(str)"), size=11, color="black")
    + scale_fill_gradientn(
        colors=["#2171b5", "#f7fbff"],
        name="Rank",
    )
    + scale_color_manual(values={False: "#333333", True: "#cccccc"}, guide=None)
    + theme_bw()
    + theme(
        axis_text_x=element_text(rotation=45, hjust=1., size=9),
        axis_text_y=element_text(size=9),
        panel_grid=element_blank(),
        figure_size=(10, 6),
    )
    + coord_equal()
    + labs(x="Factor", y="Cell Line", title=f"")
)
plot_ranks.show()

## Prediction of Pathway Stimulation

### Method Comparison (MOFA-FLEX vs Spectra vs ExpiMap)

In [ ]:
auroc_mfl_all = utils.compute_auroc_matrix(mfl_factors, adata.obs["pathway"], subsample_size=1000)
auroc_spectra_all = utils.compute_auroc_matrix(spectra_factors, adata.obs["pathway"], subsample_size=1000)
auroc_expimap_all = utils.compute_auroc_matrix(expimap_factors, adata.obs["pathway"], subsample_size=1000)

# Long-form, renamed
df_mfl_full = auroc_mfl_all.reset_index(names="Factor").melt(id_vars="Factor", var_name="Pathway", value_name="AUROC")
df_mfl_full = df_mfl_full[~df_mfl_full["Factor"].str.startswith("Factor")]
df_mfl_full["Factor"] = df_mfl_full["Factor"].replace(hallmark_map)
df_mfl_full["Method"] = "MOFA-FLEX"

df_spectra_full = auroc_spectra_all.reset_index(names="Factor").melt(id_vars="Factor", var_name="Pathway", value_name="AUROC")
df_spectra_full["Factor"] = df_spectra_full["Factor"].replace(hallmark_map)
df_spectra_full["Method"] = "Spectra"

df_expimap_full = auroc_expimap_all.reset_index(names="Factor").melt(id_vars="Factor", var_name="Pathway", value_name="AUROC")
df_expimap_full["Factor"] = df_expimap_full["Factor"].replace(hallmark_map)
df_expimap_full["Method"] = "ExpiMap"

# Top 2 factors per pathway per method
def top_factors_per_pathway(df, n=2):
    return (
        df.groupby("Pathway", observed=False)
        .apply(lambda g: g.nlargest(n, "AUROC"), include_groups=False)
        .reset_index("Pathway")[["Pathway", "Factor"]]
    )

top_mfl = top_factors_per_pathway(df_mfl_full)
top_spectra = top_factors_per_pathway(df_spectra_full)
top_expimap = top_factors_per_pathway(df_expimap_full)

# Union of all top factors — shared y-axis
union_factors = pd.concat([top_mfl, top_spectra, top_expimap])["Factor"].unique().tolist()

# Filter each full df to the union set, then combine
df_compare = pd.concat([
    df_mfl_full[df_mfl_full["Factor"].isin(union_factors)],
    df_spectra_full[df_spectra_full["Factor"].isin(union_factors)],
    df_expimap_full[df_expimap_full["Factor"].isin(union_factors)],
], ignore_index=True)

df_compare["Factor"] = pd.Categorical(df_compare["Factor"], categories=sorted(union_factors), ordered=True)
df_compare["Method"] = pd.Categorical(df_compare["Method"], categories=df_compare.Method.unique(), ordered=True)

plot = (
    ggplot(df_compare, aes(x="Pathway", y="Factor", fill="AUROC"))
    + geom_tile()
    + scale_fill_gradientn(colors=["#f7fbff", "#2171b5"], limits=[0.5, 1], name="AUROC")
    + facet_wrap("~Method", ncol=3)
    + theme_bw()
    + theme(
        axis_text_x=element_text(rotation=90, hjust=0.5, size=9),
        axis_text_y=element_text(size=9),
        strip_text=element_text(size=7, face="bold"),
        panel_grid=element_blank(),
        figure_size=(8, 4),
    )
    + labs(
        x="Stimulated Pathway",
        y="Factor",
        title="Prediction of Pathway Stimulation — All Cells",
    )
    + coord_equal()
)
plot.show()

In [ ]:
# Per cell type AUROC — same factors and order as the all-cells plot above
cell_types = adata.obs["cell_type"].unique().tolist()

dfs = []
for ct in cell_types:
    mask_ct = adata.obs["cell_type"] == ct
    ct_factors = mfl_factors.loc[mask_ct[mask_ct].index]
    ct_pathway = adata.obs.loc[mask_ct[mask_ct].index, "pathway"]
    auroc = utils.compute_auroc_matrix(ct_factors, ct_pathway, subsample_size=1000)
    df = auroc.reset_index(names="Factor").melt(id_vars="Factor", var_name="Pathway", value_name="AUROC")
    df["CellType"] = ct
    dfs.append(df)

df_mfl_ct = pd.concat(dfs, ignore_index=True)
df_mfl_ct = df_mfl_ct[~df_mfl_ct["Factor"].str.startswith("Factor")]
df_mfl_ct["Factor"] = df_mfl_ct["Factor"].replace(hallmark_map)
df_mfl_ct = df_mfl_ct[df_mfl_ct["Factor"].isin(union_factors)]
df_mfl_ct["Factor"] = pd.Categorical(df_mfl_ct["Factor"], categories=df_compare["Factor"].cat.categories, ordered=True)
df_mfl_ct["CellType"] = pd.Categorical(df_mfl_ct["CellType"], categories=sorted(cell_types), ordered=True)

plot_ct = (
    ggplot(df_mfl_ct, aes(x="Pathway", y="Factor", fill="AUROC"))
    + geom_tile()
    + scale_fill_gradientn(colors=["#f7fbff", "#2171b5"], limits=[0.5, 1], name="AUROC")
    + facet_wrap("~CellType", ncol=len(cell_types))
    + theme_bw()
    + theme(
        axis_text_x=element_text(rotation=90, hjust=0.5, size=9),
        axis_text_y=element_text(size=9),
        strip_text=element_text(size=7, face="bold"),
        panel_grid=element_blank(),
        figure_size=(8, 4),
    )
    + labs(
        x="Stimulated Pathway",
        y="Factor",
        title="Prediction of Pathway Stimulation per Cell Type — MOFA-FLEX",
    )
    + coord_equal()
)
plot_ct.show()


## UMAPs

In [ ]:
mfl_factors_adata = ad.concat(mfl_model.get_factors("anndata"))

In [ ]:
sc.pp.neighbors(mfl_factors_adata)
sc.tl.umap(mfl_factors_adata)
pd.DataFrame(mfl_factors_adata.obsm["X_umap"], columns=["UMAP1", "UMAP2"], index=adata.obs_names).to_csv("umaps/mfl.csv")
mfl_factors_adata.obsm["X_umap"] = pd.read_csv("umaps/mfl.csv", index_col=0).values

In [ ]:
umap_coords = pd.DataFrame(
    mfl_factors_adata.obsm[f"X_umap"], columns=["UMAP1", "UMAP2"], index=mfl_factors_adata.obs_names
).join(mfl_factors_adata.obs[["cell_type", "pathway", "Batch_info"]]).sample(50000)

for color, label in [("cell_type", "Cell Line"), ("pathway", "Pathway"), ("Batch_info", "Batch")]:
    plot = (
        ggplot(umap_coords, aes(x="UMAP1", y="UMAP2", color=color))
        + geom_point(size=0.1, alpha=0.1, raster=True)
        + theme_bw()
        + labs(title=f"", color=label)
        + guides(color=guide_legend(override_aes={"size": 3, "alpha": 1}))
        + theme(
            figure_size=(3.3, 3),
            axis_text_x=element_blank(),
            axis_text_y=element_blank(),
            axis_ticks_x=element_blank(),
            axis_ticks_y=element_blank(),
        )
        + coord_equal()
    )
    plot.show()